# 02 — Stationarity Analysis

**Goal:** Run ADF tests on each series, determine differencing order,
and populate the `adf_pvalue` fields in `preprocessing_config.yaml`.

**Process:**
1. Load raw series
2. For each: test level → if non-stationary, log-transform and test → difference and test
3. Record ADF p-value and differencing order
4. Write results back to preprocessing_config.yaml

**Output:** Updated `preprocessing_config.yaml` with `adf_pvalue` populated.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'backend'))

import numpy as np
import pandas as pd
import yaml
from statsmodels.tsa.stattools import adfuller

from data.ingestion import generate_dummy_series, load_series_config

In [ ]:
raw = generate_dummy_series()  # Replace with fetch_all_series() when FRED key is set
config = load_series_config()
config_by_id = {s['id']: s for s in config}

In [ ]:
def run_adf(series, name=''):
    """Run ADF test, return (statistic, p-value, is_stationary)."""
    clean = series.dropna()
    if len(clean) < 20:
        return None, None, False
    result = adfuller(clean, autolag='AIC')
    stat, pval = result[0], result[1]
    stationary = pval < 0.05
    print(f'  {name:40s} ADF stat={stat:8.3f}  p={pval:.4f}  {"✓ stationary" if stationary else "✗ non-stationary"}')
    return stat, pval, stationary

In [ ]:
results = []

for sid, df in raw.items():
    sc = config_by_id.get(sid)
    if sc is None:
        continue
    
    series = df.set_index('date')['value']
    print(f'\n--- {sid} ---')
    
    # Level test
    _, pval_level, is_stat = run_adf(series, f'{sid} (level)')
    
    if is_stat:
        results.append({'id': sid, 'differencing': 0, 'log_transform': False, 'adf_pvalue': round(pval_level, 4)})
        continue
    
    # Log + level test (if config says log_transform)
    if sc.get('log_transform', False):
        log_series = np.log(series[series > 0])
        _, pval_log, is_stat_log = run_adf(log_series, f'{sid} (log level)')
        if is_stat_log:
            results.append({'id': sid, 'differencing': 0, 'log_transform': True, 'adf_pvalue': round(pval_log, 4)})
            continue
        # Log + first difference
        diff1 = log_series.diff().dropna()
    else:
        diff1 = series.diff().dropna()
    
    _, pval_d1, is_stat_d1 = run_adf(diff1, f'{sid} (diff=1)')
    
    if is_stat_d1:
        results.append({'id': sid, 'differencing': 1, 'log_transform': sc.get('log_transform', False), 'adf_pvalue': round(pval_d1, 4)})
    else:
        # Second difference (rare)
        diff2 = diff1.diff().dropna()
        _, pval_d2, _ = run_adf(diff2, f'{sid} (diff=2)')
        results.append({'id': sid, 'differencing': 2, 'log_transform': sc.get('log_transform', False), 'adf_pvalue': round(pval_d2, 4) if pval_d2 else None})

results_df = pd.DataFrame(results)
results_df

In [ ]:
# TODO: Write adf_pvalue results back into preprocessing_config.yaml
# This should be done manually after reviewing the results above.
# Open backend/features/preprocessing_config.yaml and fill in adf_pvalue fields.

print('\n=== Copy these into preprocessing_config.yaml ===')
for _, row in results_df.iterrows():
    print(f"  {row['id']}: adf_pvalue={row['adf_pvalue']}, differencing={row['differencing']}")